# 独立 M-H 算法计算积分

> 对应代码：`MathStatsCode/code_in_notes/Chap.MCMC/MCMC_independent_integ.do`（本 notebook 中的 Stata 代码与 do 文件一致）
>
> 运行方式：通过 *PyStata* 在 Jupyter 中调用 Stata，安装与使用说明见 `notebook/PyStata.md`。

**实验目的**：以标准二元正态为工具分布，用独立 M-H 算法从目标二元正态分布抽样并计算积分。


首先启动 Stata 环境，并把工作目录切换到 `code_in_notes/Chap.MCMC`，这样 Stata 中的相对路径（如 `../datasets/...`）与同名 do 文件完全一致，图片也会输出到该目录。


In [ ]:
# 配置 PyStata：启动 Stata 并注册 %%stata 魔法命令
import os
os.chdir("../../code_in_notes/Chap.MCMC")

import stata_setup
stata_setup.config("/opt/Stata", "mp")

**独立的Metropolis-Hastings算法计算积分示例**

清空环境、关闭分页并固定随机数种子（沿用原 Python 代码中的 19880505）；`local M = 10000` 设定抽样次数。


In [ ]:
%%stata
clear all
set more off
set seed 19880505


**设定参数**

这一大段用**独立的 Metropolis-Hastings 算法**从目标分布 π（二元正态，均值 (0.5,-0.1)、方差 1/180 与 1/90）中抽样，再计算积分 π/(90√2)·E_π[h(x)]（h 为定义域指示函数）。各部分含义：

- `pai(x)`：目标密度——二元正态密度函数；`h(x)`：被积函数（定义域指示）；`q(x)`：工具密度——**标准**二元正态（与目标分布形状不同、但覆盖其支撑）；`q_sampler()`：从工具分布抽样（Mata 的 `rnormal(1,2,0,1)` 产生一行两列的标准正态随机数）；
- `MH_independent(...)`：独立 MH 算法——每个候选点都从固定工具分布独立抽取；接受概率 ρ=min{1, π(y)q(x)/(π(x)q(y))}；被拒绝时链保持原状态；
- `X = MH_independent(...)`：执行抽样，X 是 M×2 矩阵，每一行是链的一个状态；
- 计算积分：对每一行求 h 值（Mata 不能用向量化自动映射任意函数，用循环逐行计算），乘上正态常数 π/(90√2) 后取均值 `printf` 输出结果。

为什么用 MCMC 算这个积分？因为可以直接从 π 抽样时积分 = 常数×E[h]；本例虽然也可以直接抽正态，但目的是**演示算法**——用标准正态作为工具分布、通过接受-拒绝校正到目标分布。


In [ ]:
%%stata
local M = 10000

mata:
// 目标密度：二元正态分布，mu=(0.5,-0.1)，方差分别为1/180和1/90
real scalar pai(real rowvector x)
{
	real scalar s1, s2
	s1 = 1/180
	s2 = 1/90
	return(1/(2*pi()*sqrt(s1)*sqrt(s2))*exp(-(x[1]-0.5)^2/(2*s1) - (x[2]+0.1)^2/(2*s2)))
}

// 被积函数：定义域指示函数
real scalar h(real rowvector x)
{
	return(x[1]>=-1 && x[1]<=1 && x[2]>=-1 && x[2]<=1)
}

// 工具密度：标准二元正态分布
real scalar q(real rowvector x)
{
	return(1/(2*pi())*exp(-(x[1]^2+x[2]^2)/2))
}

// 从工具分布中抽样
real rowvector q_sampler()
{
	return(rnormal(1, 2, 0, 1))
}

// 独立的MCMC算法，输入：
//    N_samples  : 抽样次数
//      pai(x)   : 目标密度函数
//      q(y)     : 工具密度函数
//   q_sampler   : 从q中抽样的函数
//      x0       : 初始值
real matrix MH_independent(real scalar N_samples, pointer pai, pointer q,
                           pointer q_sampler, real rowvector x0)
{
	real matrix X
	real rowvector x, y
	real scalar i, rho, u
	X = J(N_samples, length(x0), .)
	x = x0
	for (i=1; i<=N_samples; i++) {
		y = (*q_sampler)()
		rho = min((1, (*pai)(y)*(*q)(x)/((*pai)(x)*(*q)(y))))
		u = runiform(1, 1)
		if (u <= rho) {
			x = y
		}
		X[i, .] = x
	}
	return(X)
}

// 使用独立的Metropolis-Hastings算法从pai(x)中抽样
X = MH_independent(`M', &pai(), &q(), &q_sampler(), (0,0))

// 计算积分：I = pi/(90*sqrt(2)) * E_pi[h(x)]
H = J(`M', 1, .)
for (i=1; i<=`M'; i++) {
	H[i] = h(X[i, .])
}
integral = pi()/(90*sqrt(2))*mean(H)
printf("Integral = %f\n", integral)
end
